# Preprocessing

## The Two Most Common Preprocessing Steps
### 1. Normalization (Scaling Features)
Just like StandardScaler in scikit-learn — subtract mean, divide by standard deviation.
```python
# Classical ML way (scikit-learn)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PyTorch way — do it with tensors
mean = X_train.mean(dim=0)   # Mean of each feature column
std  = X_train.std(dim=0)    # Std of each feature column

X_train_scaled = (X_train - mean) / std
X_val_scaled   = (X_val   - mean) / std   # Use TRAIN mean/std on val too!
#                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#                          Same rule as scikit-learn — never fit on val data
```

## 2. Preprocessing Inside the Dataset
The cleaner, more scalable pattern — bake normalization into the Dataset itself:
```python
class MyDataset(Dataset):
    def __init__(self, X, y, mean=None, std=None):
        self.y = y

        # Compute mean/std if not provided (i.e. training set)
        self.mean = mean if mean is not None else X.mean(dim=0)
        self.std  = std  if std  is not None else X.std(dim=0)

        # Normalize immediately
        self.X = (X - self.mean) / self.std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]
```

Then use training stats for both datasets:
```python
# Fit stats on training data only
train_dataset = MyDataset(X_train, y_train)

# Pass those same stats to val — never recompute on val
val_dataset = MyDataset(X_val, y_val, mean=train_dataset.mean, std=train_dataset.std)
```

# Full Working Example

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

In [2]:
class MyDataset(Dataset):
    def __init__(self, X, y, mean=None, std=None):
        self.y = y 
        self.mean = mean if mean is not None else X.mean(dim=0)
        self.std  = std  if std  is not None else X.std(dim=0)
        self.X = (X - self.mean) / self.std     # Normalize on init
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]    

## Raw data (unscaled, different ranges per feature)

In [3]:
X = torch.cat([
    torch.randn(100, 2),           # Features 1-2: small range
    torch.randn(100, 2) * 100      # Features 3-4: large range (unscaled!)
], dim=1)
y = torch.randint(0, 2, (100,))

In [4]:
X[0], y[0]

(tensor([  1.9269,   1.4873, 130.3207,  48.7868]), tensor(0))

## Train/val split

In [5]:
X_train, y_train = X[:80], y[:80]
X_val,   y_val   = X[80:], y[80:]

## Build datasets — val reuses train stats

In [6]:
train_dataset = MyDataset(X_train, y_train)
val_dataset   = MyDataset(X_val, y_val,
                          mean=train_dataset.mean,
                          std=train_dataset.std)

## DataLoaders

In [7]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

## Model

In [8]:
model = nn.Sequential(
    nn.Linear(4, 32), 
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

## training loop

In [9]:
for epoch in range(5):
    model.train()
    total_train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            total_val_loss += criterion(model(X_batch), y_batch).item()
    
    print(f"Epoch {epoch+1} | "
          f"Train Loss: {total_train_loss/len(train_loader):.4f} | "
          f"Val Loss: {total_val_loss/len(val_loader):.4f}")

Epoch 1 | Train Loss: 0.7108 | Val Loss: 0.6608
Epoch 2 | Train Loss: 0.6608 | Val Loss: 0.6431
Epoch 3 | Train Loss: 0.6560 | Val Loss: 0.6298
Epoch 4 | Train Loss: 0.6400 | Val Loss: 0.6409
Epoch 5 | Train Loss: 0.6282 | Val Loss: 0.6616


The Golden Rule of Preprocessing

Always compute mean/std on training data only.
Then apply those same values to validation and test data.